In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/xyz2005/notebook-3/train_features (1).pt
/kaggle/input/datasets/xyz2005/notebook-3/train_features.pt
/kaggle/input/datasets/xyz2005/notebook-3/test_features.pt
/kaggle/input/datasets/xyz2005/notebook-3/valid_features.pt
/kaggle/input/datasets/xyz2005/notebook-3/test_reports.csv
/kaggle/input/datasets/xyz2005/notebook-3/valid_reports.csv
/kaggle/input/datasets/xyz2005/notebook-3/train_reports.csv
/kaggle/input/datasets/xyz2005/concept-labels/valid_concepts.csv
/kaggle/input/datasets/xyz2005/concept-labels/test_concepts.csv
/kaggle/input/datasets/xyz2005/concept-labels/train_concepts.csv


In [2]:
!pip install -q torch torchvision pandas scikit-learn matplotlib

In [3]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [5]:
train_features = torch.load(
"/kaggle/input/datasets/xyz2005/notebook-3/train_features.pt"
)

valid_features = torch.load(
"/kaggle/input/datasets/xyz2005/notebook-3/valid_features.pt"
)

test_features = torch.load(
"/kaggle/input/datasets/xyz2005/notebook-3/test_features.pt"
)

In [6]:
train_df = pd.read_csv(
"/kaggle/input/datasets/xyz2005/concept-labels/train_concepts.csv"
)

valid_df = pd.read_csv(
"/kaggle/input/datasets/xyz2005/concept-labels/valid_concepts.csv"
)

test_df = pd.read_csv(
"/kaggle/input/datasets/xyz2005/concept-labels/test_concepts.csv"
)

In [7]:
LABEL_COLUMNS = [

"Pneumonia",

"Cardiomegaly",

"Pleural Effusion",

"Atelectasis",

"Edema",

"Pneumothorax",

"Consolidation",

"Lung Opacity",

"Nodule",

"Mass"

]

In [8]:
train_labels = torch.tensor(
train_df[LABEL_COLUMNS].values,
dtype=torch.float32
)

valid_labels = torch.tensor(
valid_df[LABEL_COLUMNS].values,
dtype=torch.float32
)

test_labels = torch.tensor(
test_df[LABEL_COLUMNS].values,
dtype=torch.float32
)

In [9]:
class ConceptDataset(Dataset):

    def __init__(
        self,
        features,
        labels
    ):

        self.features = features
        self.labels = labels

    def __len__(self):

        return len(self.features)

    def __getitem__(self, idx):

        return (

            self.features[idx],

            self.labels[idx]

        )

In [10]:
train_loader = DataLoader(

ConceptDataset(
train_features,
train_labels
),

batch_size=32,

shuffle=True

)

valid_loader = DataLoader(

ConceptDataset(
valid_features,
valid_labels
),

batch_size=32

)

In [11]:
class ConceptClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(1408,512),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(512,256),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256,10)

        )

    def forward(self,x):

        return self.network(x)

In [12]:
model = ConceptClassifier().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(

model.parameters(),

lr=1e-4

)

In [13]:
history=[]

EPOCHS=60

for epoch in range(EPOCHS):

    model.train()

    total_loss=0

    for features,labels in train_loader:

        features = features.float().to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()

        outputs=model(features)

        loss=criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

    avg_loss=total_loss/len(train_loader)

    history.append(avg_loss)

    print(

        f"Epoch {epoch+1}"

        f" Loss:{avg_loss:.4f}"

    )

Epoch 1 Loss:0.3976
Epoch 2 Loss:0.3695
Epoch 3 Loss:0.3663
Epoch 4 Loss:0.3667
Epoch 5 Loss:0.3636
Epoch 6 Loss:0.3641
Epoch 7 Loss:0.3618
Epoch 8 Loss:0.3623
Epoch 9 Loss:0.3576
Epoch 10 Loss:0.3591
Epoch 11 Loss:0.3570
Epoch 12 Loss:0.3589
Epoch 13 Loss:0.3566
Epoch 14 Loss:0.3545
Epoch 15 Loss:0.3522
Epoch 16 Loss:0.3529
Epoch 17 Loss:0.3509
Epoch 18 Loss:0.3498
Epoch 19 Loss:0.3504
Epoch 20 Loss:0.3511
Epoch 21 Loss:0.3488
Epoch 22 Loss:0.3480
Epoch 23 Loss:0.3471
Epoch 24 Loss:0.3473
Epoch 25 Loss:0.3437
Epoch 26 Loss:0.3471
Epoch 27 Loss:0.3448
Epoch 28 Loss:0.3440
Epoch 29 Loss:0.3452
Epoch 30 Loss:0.3460
Epoch 31 Loss:0.3445
Epoch 32 Loss:0.3432
Epoch 33 Loss:0.3433
Epoch 34 Loss:0.3400
Epoch 35 Loss:0.3401
Epoch 36 Loss:0.3409
Epoch 37 Loss:0.3408
Epoch 38 Loss:0.3405
Epoch 39 Loss:0.3385
Epoch 40 Loss:0.3359
Epoch 41 Loss:0.3343
Epoch 42 Loss:0.3367
Epoch 43 Loss:0.3375
Epoch 44 Loss:0.3363
Epoch 45 Loss:0.3368
Epoch 46 Loss:0.3367
Epoch 47 Loss:0.3343
Epoch 48 Loss:0.3346
E

In [14]:
torch.save(

model.state_dict(),

"/kaggle/working/concept_classifier.pth"

)

In [15]:
history_df=pd.DataFrame(

{

"loss":history

}

)

history_df.to_csv(

"/kaggle/working/training_history.csv",

index=False

)

In [16]:
model.eval()

predictions=[]

ground_truth=[]

with torch.no_grad():

    for features,labels in valid_loader:

        features=features.float().to(device)

        outputs=model(features)

        preds=torch.sigmoid(outputs)

        preds=(preds>0.5).float()

        predictions.extend(preds.cpu().numpy())

        ground_truth.extend(labels.numpy())

In [17]:
predictions=np.array(predictions)

ground_truth=np.array(ground_truth)

accuracy=(predictions==ground_truth).mean()

print("Validation Accuracy:",accuracy)

Validation Accuracy: 0.8704453441295547


In [18]:
torch.save(
    model.state_dict(),
    "/kaggle/working/concept_classifier.pth"
)

history_df.to_csv(
    "/kaggle/working/training_history.csv",
    index=False
)